# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Arslan
**Student ID:**
**Group:**
**Repository:** https://github.com/ars512/css4007-hw3

## AI tool disclosure

I used Claude Code (Claude Sonnet 5.5) to write the notebook code and the Part A working, and to draft the conceptual answers. The notebook was executed top to bottom (`gpt-5.6-luna` answered every call; no fallback model was used); the B3, C2, C3 and final-question analysis below was written from the printed outputs of that run and quotes its ids, scores and token counts.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

Tokenisation: lowercase, split on spaces and hyphens, drop punctuation, no stop words, no stemming.

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.333**

**Query terms:** how, does, rag, retrieve, information

**TF, DF and IDF of each query term** (N = 3): idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(7) = 1.9459 for every term, because n(t) = 0.

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document:**

> Every term has f(t,d) = 0, so every term contributes idf · 0 · 2.5 / (0 + …) = 0.
> - D1 = 0 + 0 + 0 + 0 + 0 = 0
> - D2 = 0 + 0 + 0 + 0 + 0 = 0
> - D3 = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = 25 / 3 = **8.333** (stemming does not change lengths)

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | ln((3 − 1 + 0.5)/(1 + 0.5)) = ln(1.6667) = 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document**, worked:

> Only `retriev` has f > 0, and only in D2. Length norm for D2: 1 − 0.75 + 0.75 · 10 / 8.333 = 0.25 + 0.9 = 1.15, so k1 · 1.15 = 1.725.
> - D2: 0.5108 · 2 · 2.5 / (2 + 1.725) = 2.5541 / 3.725 = **0.6857**
> - D1 = 0, D3 = 0 (no query term present)

| Document | BM25 Score | Rank |
|---|---:|---:|
| D2 | 0.6857 | 1 |
| D1 | 0.0000 | 2 (tie) |
| D3 | 0.0000 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a)** None. All three scores are 0, so there is no first document, only a three-way tie. This is not an arithmetic mistake: after tokenising, the query token `retrieve` never appears in any document. D2 has `retrieves` and `retrieval`, which are different strings for BM25, and `how`, `does`, `rag`, `information` appear nowhere. The check holds: BM25 matches strings, not meanings.
>
> **(b)** D2, with 0.6857. Stemming maps `retrieve`, `retrieves` and `retrieval` to `retriev`, so D2 matches the query (tf = 2). It is the only match, and its idf is small (0.51) because one of three documents contains it. The "obvious" terms `rag` and `information` still contribute nothing: D2 says "Retrieval-Augmented Generation", not "RAG".

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | √(0.49+0.01+0.16) = √0.66 = 0.8124 | 0.78 / (0.9644·0.8124) = 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | √(0.81+0.09+0.36) = √1.26 = 1.1225 | 1.08 / (0.9644·1.1225) = 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | √(0.01+0.64+0.04) = √0.69 = 0.8307 | 0.34 / (0.9644·0.8307) = 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D2 | 0.9977 | 1 |
| D1 | 0.9956 | 2 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 (0.9977), by a hair over D1 (0.9956). D3 is far away (0.4244).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Partly. Against A1 (b) the top document agrees (D2 first), but cosine also separates D1 from D3 (D2 > D1 > D3), whereas BM25 (b) leaves D1 and D3 tied at 0. Against A1 (a) it disagrees completely: BM25 as written found nothing (three-way tie), while cosine still ranks D2 first.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 needs the same token. The query says `retrieve`, `RAG`, `information`; D2 says `retrieves`, `Retrieval-Augmented Generation`, `documents`. No shared string (without stemming), so BM25 sees nothing. An embedding model can place "RAG" near "Retrieval-Augmented Generation" and "retrieve" near "retrieves"/"retrieval", so D2 lands close to the query vector. The flip side: D1 ("generate text and answer questions") also gets a very high cosine (0.9956) although it says nothing about retrieval, because embeddings are fuzzy; BM25 would never give it a score. (The vectors here are given, not computed, so this is an assumption about what a real model does.)

Check of the Part A arithmetic (the working above is by hand).

In [1]:
import math, re
docs = ["Large language models can generate text and answer questions.",
        "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
        "Embeddings represent text as numerical vectors."]
query = "How does RAG retrieve information?"
stems = {'retrieve': 'retriev', 'retrieves': 'retriev', 'retrieval': 'retriev', 'generate': 'gener', 'generating': 'gener',
         'generation': 'gener', 'information': 'inform', 'does': 'doe'}
def tok(s, stem):
    t = re.findall(r"[a-z0-9]+", s.lower().replace('-', ' '))
    return [stems.get(x, x) for x in t] if stem else t
for stem in (False, True):
    D = [tok(d, stem) for d in docs]; Q = tok(query, stem); N = len(D); avgdl = sum(map(len, D)) / N
    scores = [0.0] * N
    for t in Q:
        n = sum(t in d for d in D); idf = math.log((N - n + .5) / (n + .5))
        for i, d in enumerate(D):
            f = d.count(t)
            if f: scores[i] += idf * f * 2.5 / (f + 1.5 * (.25 + .75 * len(d) / avgdl))
    print("stemming" if stem else "as written", "| lengths", [len(d) for d in D], "avgdl", round(avgdl, 3), "| scores", [round(s, 4) for s in scores])
qv = [.8, .2, .5]
for name, v in [("D1", [.7, .1, .4]), ("D2", [.9, .3, .6]), ("D3", [.1, .8, .2])]:
    dot = sum(a * b for a, b in zip(qv, v)); mq = math.sqrt(sum(a * a for a in qv)); mdv = math.sqrt(sum(a * a for a in v))
    print(name, "dot", round(dot, 4), "|q|", round(mq, 4), "|d|", round(mdv, 4), "cos", round(dot / (mq * mdv), 4))

as written | lengths [9, 10, 6] avgdl 8.333 | scores [0.0, 0.0, 0.0]
stemming | lengths [9, 10, 6] avgdl 8.333 | scores [0.0, 0.6857, 0.0]
D1 dot 0.78 |q| 0.9644 |d| 0.8124 cos 0.9956
D2 dot 1.08 |q| 0.9644 |d| 1.1225 cos 0.9977
D3 dot 0.34 |q| 0.9644 |d| 0.8307 cos 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import os, json, re, warnings
import numpy as np
from dotenv import load_dotenv
load_dotenv()
warnings.filterwarnings("ignore")

corpus = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf8")]
ids = [c["id"] for c in corpus]
texts = [c["text"] for c in corpus]
byid = {c["id"]: c for c in corpus}
queries = json.load(open("data/queries.json", encoding="utf8"))["queries"]
qa = json.load(open("data/qa.json", encoding="utf8"))["questions"]
print(len(corpus), "chunks;", len(queries), "queries;", len(qa), "questions")

74 chunks; 8 queries; 6 questions


## B1 — BM25 with `rank_bm25`

**My tokenisation:** lowercase, `re.findall(r"\w+", text.lower())` (Unicode-aware, so Kazakh and Russian words survive; `GR-305` becomes the two tokens `gr` and `305`), **no stemming, no stop-word removal**. `BM25Okapi(k1=1.5, b=0.75)`. Consequences: no morphology (`retrieves` ≠ `retrieve`), and a Kazakh query can only match Kazakh chunks, because it shares no word with the English ones.

In [3]:
from rank_bm25 import BM25Okapi
def tokenize(s): return re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def first_line(c, n=80):
    return c["text"].split("\n")[0][:n]
def bm25_top(q, k=3):
    s = bm25.get_scores(tokenize(q)); o = np.argsort(-s)[:k]
    return [(ids[i], float(s[i])) for i in o]

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("google/embeddinggemma-300m", token=os.getenv("HF_TOKEN"))   # default float32
doc_emb = model.encode_document(texts, show_progress_bar=False)   # computed once, kept in a numpy array
print("embedding matrix:", doc_emb.shape)
def emb_top(q, k=3):
    s = model.similarity(model.encode_query(q), doc_emb)[0].numpy(); o = np.argsort(-s)[:k]
    return [(ids[i], float(s[i])) for i in o]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:  14%|█▍        | 44/314 [00:00<00:00, 431.80it/s]

Loading weights:  28%|██▊       | 88/314 [00:00<00:00, 381.05it/s]

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1332.96it/s]

embedding matrix: (74, 768)


## My own queries

Written, with their `relevant` ids, **before** running them.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which form is the household income declaration, GR-210? | GO-07, GO-03, GO-10, GO-13 | **BM25.** The rare exact token `210` (with `gr`) occurs in only a few chunks, so BM25 gives it a high idf; an embedding model tends to blur the form codes GR-104/GR-210/GR-305 into "a grant form". |
| Q-10 | Can I take a year off university and keep my money? | GO-18 | **Embeddings.** The handbook says "academic leave" and "payments stop"; the query says "year off" and "keep my money". Only a paraphrase, almost no shared words. |

In [5]:
my_queries = [
    {"id": "Q-09", "text": "Which form is the household income declaration, GR-210?", "relevant": ["GO-07", "GO-03", "GO-10", "GO-13"]},
    {"id": "Q-10", "text": "Can I take a year off university and keep my money?", "relevant": ["GO-18"]},
]
all_queries = queries + my_queries

## Results for all ten or more queries: top 3 from each retriever

In [6]:
results = {}
for q in all_queries:
    b, e = bm25_top(q["text"]), emb_top(q["text"])
    rel = set(q["relevant"])
    results[q["id"]] = {"bm25": b, "emb": e, "bm25_hit": any(i in rel for i, _ in b), "emb_hit": any(i in rel for i, _ in e)}
    print(f"\n=== {q['id']}: {q['text']}\n    relevant: {q['relevant']}")
    for name, top in (("BM25", b), ("EmbeddingGemma", e)):
        print(f"  {name}  hit@3 = {any(i in rel for i, _ in top)}")
        for r, (i, s) in enumerate(top, 1):
            print(f"    {r}. {i}  {s:8.4f}  {'*' if i in rel else ' '} {first_line(byid[i])}")


=== Q-01: What is form GR-305 used for?
    relevant: ['GO-08', 'GO-12', 'GO-23']
  BM25  hit@3 = True
    1. GO-08   15.8255  * Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f
    2. GO-12    8.1953  * A student who disagrees with a decision may appeal within 14 calendar days of th
    3. GO-06    8.0700    Form GR-104 is the application form for the need-based study grant. It is the on
  EmbeddingGemma  hit@3 = True
    1. GO-08    0.6955  * Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f
    2. GO-12    0.4609  * A student who disagrees with a decision may appeal within 14 calendar days of th
    3. GO-23    0.4505  * Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в

=== Q-02: My family is poor. Is there financial help for students like me?
    relevant: ['GO-01', 'GO-02', 'GO-03']
  BM25  hit@3 = False
    1. KZ-17    8.7262    The lake currently covers about 16,400 km2 (6,300 sq mi).


=== Q-03: Өтінішті қашанға дейін тапсыру керек?
    relevant: ['GO-09', 'GO-14']
  BM25  hit@3 = True
    1. GO-14   10.6979  * 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік по
    2. KZ-01    0.0000    Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situa
    3. KZ-03    0.0000    Kazakhstan has been inhabited since the Paleolithic era. In antiquity, various n
  EmbeddingGemma  hit@3 = True
    1. GO-14    0.3992  * 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік по
    2. GO-09    0.3502  * Applications for the 2026 grant open on 1 September 2026 and close on 15 October
    3. GO-11    0.3007    If a required document is missing, the Grant Office does not refuse the applicat



=== Q-04: What is the minimum GPA for the grant?
    relevant: ['GO-02']
  BM25  hit@3 = False
    1. GO-05   12.7798    ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for th
    2. GO-17   10.5176    The grant continues only while the student keeps up their studies. If the cumula
    3. GO-19   10.1360    The grant budget is fixed each year. If there are more qualifying applicants tha
  EmbeddingGemma  hit@3 = True
    1. GO-02    0.6652  * An applicant qualifies for the 2026 grant when all three conditions hold at the 
    2. GO-05    0.6210    ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for th
    3. GO-17    0.6134    The grant continues only while the student keeps up their studies. If the cumula

=== Q-05: When was Astana renamed Nur-Sultan, and why?
    relevant: ['KZ-05']
  BM25  hit@3 = True
    1. KZ-05   15.7315  * Astana is the capital city of Kazakhstan. With a population of 1,7 million withi
    2. KZ-06    6.4431 


=== Q-06: Where did the Soviet Union test its nuclear weapons?
    relevant: ['KZ-37', 'KZ-38']
  BM25  hit@3 = True
    1. KZ-37   19.0904  * The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: Семипалатинск-21; Kaza
    2. KZ-38   17.1677  * The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 1949 until 19
    3. KZ-03    9.3523    Kazakhstan has been inhabited since the Paleolithic era. In antiquity, various n
  EmbeddingGemma  hit@3 = True
    1. KZ-37    0.5960  * The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: Семипалатинск-21; Kaza
    2. KZ-38    0.5102  * The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 1949 until 19
    3. KZ-10    0.3315    The Baikonur Cosmodrome is a spaceport operated by Russia within Kazakhstan. Loc



=== Q-07: Can the grant be sent to my mother's card?
    relevant: ['GO-15']
  BM25  hit@3 = False
    1. GO-21   14.6517    The Grant Office is in room 214 of the main building and is open Monday to Frida
    2. GO-16    9.1604    The need-based grant can be held together with a merit scholarship, but not toge
    3. GO-01    9.0532    The need-based study grant 2026 is run by the Grant Office for full-time undergr
  EmbeddingGemma  hit@3 = True
    1. GO-15    0.4216  * Грант выплачивается двумя равными частями — в октябре и в феврале — на банковски
    2. GO-18    0.4171    A student who goes on academic leave keeps their grant decision, but payments st
    3. GO-01    0.4105    The need-based study grant 2026 is run by the Grant Office for full-time undergr

=== Q-08: If my grades drop in the autumn, do I still get paid in February?
    relevant: ['GO-17']
  BM25  hit@3 = True
    1. GO-17   17.1682  * The grant continues only while the student keeps up their studies. If the cumul


=== Q-09: Which form is the household income declaration, GR-210?
    relevant: ['GO-07', 'GO-03', 'GO-10', 'GO-13']


  BM25  hit@3 = True
    1. GO-07   26.1293  * Form GR-210 is the household income declaration. It lists every adult living in 
    2. GO-03   16.2581  * The income band is set from the monthly household income per person, declared on
    3. GO-10   13.5956  * Proof of income must accompany form GR-210. Accepted: a salary certificate from 
  EmbeddingGemma  hit@3 = True
    1. GO-07    0.7858  * Form GR-210 is the household income declaration. It lists every adult living in 
    2. GO-10    0.6001  * Proof of income must accompany form GR-210. Accepted: a salary certificate from 
    3. GO-13    0.5633  * A student whose household income changes during the year must report it within 3

=== Q-10: Can I take a year off university and keep my money?
    relevant: ['GO-18']
  BM25  hit@3 = False
    1. GO-16    9.4879    The need-based grant can be held together with a merit scholarship, but not toge
    2. GO-13    9.3396    A student whose household income changes during the year must r

## B3 — comparison

The table is printed by the next cell from the actual runs (not typed by hand).

In [7]:
print(f"{'Query':6} | {'BM25 top 3':26} | hit | {'Embedding top 3':26} | hit")
for q in all_queries:
    r = results[q["id"]]
    print(f"{q['id']:6} | {', '.join(i for i, _ in r['bm25']):26} | {'Y' if r['bm25_hit'] else 'N':3} | {', '.join(i for i, _ in r['emb']):26} | {'Y' if r['emb_hit'] else 'N'}")
print("hit@3 total: BM25", sum(r["bm25_hit"] for r in results.values()), "/", len(results),
      "| Embedding", sum(r["emb_hit"] for r in results.values()), "/", len(results))

Query  | BM25 top 3                 | hit | Embedding top 3            | hit
Q-01   | GO-08, GO-12, GO-06        | Y   | GO-08, GO-12, GO-23        | Y
Q-02   | KZ-17, GO-20, KZ-25        | N   | GO-01, GO-10, GO-20        | Y
Q-03   | GO-14, KZ-01, KZ-03        | Y   | GO-14, GO-09, GO-11        | Y
Q-04   | GO-05, GO-17, GO-19        | N   | GO-02, GO-05, GO-17        | Y
Q-05   | KZ-05, KZ-06, KZ-11        | Y   | KZ-05, KZ-06, KZ-03        | Y
Q-06   | KZ-37, KZ-38, KZ-03        | Y   | KZ-37, KZ-38, KZ-10        | Y
Q-07   | GO-21, GO-16, GO-01        | N   | GO-15, GO-18, GO-01        | Y
Q-08   | GO-17, GO-05, GO-04        | Y   | GO-17, GO-18, GO-15        | Y
Q-09   | GO-07, GO-03, GO-10        | Y   | GO-07, GO-10, GO-13        | Y
Q-10   | GO-16, GO-13, KZ-07        | N   | GO-18, GO-17, GO-22        | Y
hit@3 total: BM25 6 / 10 | Embedding 10 / 10


**Both retrievers agree:** **Q-05** ("When was Astana renamed Nur-Sultan, and why?"). Both put KZ-05 first (BM25 15.73, EmbeddingGemma 0.6514) and KZ-06 second. It was easy because the query contains rare, distinctive words (`Astana`, `Nur-Sultan`) that occur in essentially one chunk, so the lexical match and the semantic match point to the same place; the rest of the corpus is about other topics. Q-06 (Semipalatinsk: KZ-37, KZ-38 in both) is the same story.

**BM25 did better:** on hit@3 there is **no query where BM25 beat EmbeddingGemma** (BM25 6/10, embeddings 10/10), and my prediction for Q-09 (BM25 wins) came out as a tie: both hit, both put GO-07 first. The evidence for BM25's strength is in the *margin*: for Q-09 it matched the exact tokens `gr` and `210` and gave GO-07 a score of 26.13 against 16.26 for the runner-up (1.6x); EmbeddingGemma's gap was 0.7858 vs 0.6001. In Q-09 BM25 returned GO-07, GO-03, GO-10, which all literally contain `GR-210`, whereas the third embedding hit was GO-13 (which only talks about income changes). Likewise for Q-01 BM25's GO-08 scored 15.83 against 8.20 for the next chunk, driven by the token `305`. So BM25 is sharper on exact form codes, but it was not more accurate here.

**Embeddings did better:** four clear cases where BM25 missed entirely and embeddings hit.
- **Q-10** ("year off university and keep my money") needs GO-18 ("academic leave ... payments stop"). BM25 top 3: GO-16, GO-13, KZ-07, all matching stray words; embeddings had GO-18 first (0.5780). BM25 could not match a **paraphrase**: no word of "year off" occurs in the chunk.
- **Q-02** ("My family is poor ... financial help"): BM25 returned KZ-17 (Aral Sea), GO-20, KZ-25 (beshbarmak), because of function words like "my", "is", "for"; the handbook says "low-income household" and "grant", a **synonym** gap. Embeddings: GO-01 first.
- **Q-07** ("sent to my mother's card?"): the answer is in GO-15, which is written in **Russian** (the Russian text says transfer to a parent's account is impossible). BM25 had no shared token (top 1 was GO-21, the office opening hours); embeddings put GO-15 first (0.4216).
- **Q-03** (Kazakh query): BM25 found the Kazakh chunk GO-14 but its remaining slots had score 0.0000 (KZ-01, KZ-03), so it missed the English GO-09 with the same deadline; embeddings returned GO-14 and GO-09 (0.3992, 0.3502), i.e. they matched **across languages**.

**Q-04 (minimum GPA):** BM25 put **GO-05 first (12.78)**: the chunk titled "Archived: the 2025 scheme (no longer in force)", which contains "minimum GPA" literally and says 3.0. The correct GO-02 ("Who qualifies", 2.67) is not in BM25's top 3, so its hit@3 is False. EmbeddingGemma got GO-02 first (0.6652) but GO-05 is a close second (0.6210): it also found the archived rule near-identical in meaning. The title says "no longer in force", but a retriever does not read the chunk the way a human does.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [8]:
import tiktoken
from openai import OpenAI
client = OpenAI()   # reads OPENAI_API_KEY from the environment (.env); the key is never printed
MODEL = "gpt-5.6-luna"          # the model required by the assignment
FALLBACK_MODEL = "gpt-4o-mini"   # used only if the account has no access to MODEL (see the printed notice)
enc = tiktoken.get_encoding("o200k_base")
ntok = lambda s: len(enc.encode(s))

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def context_block(chunk_ids):
    return "\n\n".join(f"[{n}] {byid[i]['title']}\n{byid[i]['text']}" for n, i in enumerate(chunk_ids, 1))
def build_messages(question, chunk_ids=None):
    if chunk_ids is None:     # LLM only: the bare question, no context
        return [{"role": "user", "content": question}]
    user = f"Context:\n{context_block(chunk_ids)}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
def ask(messages):
    global MODEL
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages)
    except Exception as e:
        if MODEL == FALLBACK_MODEL or "model" not in str(e).lower():
            raise
        print(f"NOTICE: {MODEL} is not available for this key ({type(e).__name__}); using {FALLBACK_MODEL} for all calls from now on")
        MODEL = FALLBACK_MODEL
        r = client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content, r.usage.prompt_tokens
qa_by = {q["id"]: q for q in qa}

## C1 — the complete prompt, printed before it is sent

Question used: **C-01** (income band 2 amount and payment dates), retrieved with EmbeddingGemma, top 3.

In [9]:
q = qa_by["C-01"]
top_ids = [i for i, _ in emb_top(q["question"], 3)]
msgs = build_messages(q["question"], top_ids)
for m in msgs:
    print(f"----- role: {m['role']} -----\n{m['content']}\n")

ctx = context_block(top_ids)
t_sys, t_q, t_ctx = ntok(SYSTEM), ntok(q["question"]), ntok(ctx)
t_all = ntok(msgs[0]["content"]) + ntok(msgs[1]["content"])
ans, usage_tokens = ask(msgs)
print("retrieved chunk ids:", top_ids)
print("system:", t_sys, "| question:", t_q, "| retrieved context:", t_ctx, "| sum of the three parts:", t_sys + t_q + t_ctx)
print("tiktoken total of both messages (incl. my 'Context:' / 'Question:' labels):", t_all)
print("usage.prompt_tokens:", usage_tokens)
print("chunks:", len(top_ids), "| tokens per chunk:", [ntok(byid[i]['text']) for i in top_ids],
      "| context size:", t_ctx, "tokens,", len(ctx), "characters")
print("\nANSWER:", ans)

----- role: system -----
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

----- role: user -----
Context:
[1] How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although t

retrieved chunk ids: ['GO-04', 'GO-03', 'GO-15']
system: 44 | question: 18 | retrieved context: 302 | sum of the three parts: 364
tiktoken total of both messages (incl. my 'Context:' / 'Question:' labels): 368
usage.prompt_tokens: 378
chunks: 3 | tokens per chunk: [90, 104, 85] | context size: 302 tokens, 1197 characters

ANSWER: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 302 |
| **Total input** (system + question + context = 364; tiktoken count of the two message strings including my "Context:" / "Question:" labels = **368**) | 368 |
| **`usage.prompt_tokens` from the response** | **378** |

- Number of retrieved chunks: 3 (GO-04, GO-03, GO-15)
- Tokens per chunk: 90, 104, 85
- Total context size: 302 tokens / 1197 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> 378 vs 368: the API reports 10 more tokens. The chat format wraps every message in special tokens (role markers and delimiters, and the reply is primed with an assistant header), none of which are in the raw strings tiktoken counted. The three separate parts (44 + 18 + 302 = 364) are also 4 below my own 368 total, because the labels "Context:", "Question:" are counted in the total but not in the parts. As in HW1, the server's count is the one that is billed.

**What exactly does retrieval add to the model's input?**

> Literally 302 tokens (1197 characters) of text: the three chunks, each preceded by `[n]` and its title, pasted into the user message. In this run block [1] "How much the grant pays" contains the answer (150,000 tenge, October and February). Nothing else is added: no document, no index, no vectors, no tool. The model's answer ("150,000 tenge ... two equal instalments ... October ... February [1]") is a continuation of that text, which is why it could cite [1].

## C2 — three systems, six questions

In [10]:
c2 = {}
for q in qa:
    b_ids = [i for i, _ in bm25_top(q["question"], 3)]
    e_ids = [i for i, _ in emb_top(q["question"], 3)]
    c2[q["id"]] = {
        "llm": ask(build_messages(q["question"]))[0],
        "bm25": ask(build_messages(q["question"], b_ids))[0], "bm25_ids": b_ids,
        "emb": ask(build_messages(q["question"], e_ids))[0], "emb_ids": e_ids,
    }
    r = c2[q["id"]]
    print(f"\n=== {q['id']}: {q['question']}\nREFERENCE: {q['reference']}")
    print(f"--- LLM only:\n{r['llm']}\n--- BM25 RAG {b_ids}:\n{r['bm25']}\n--- Embedding RAG {e_ids}:\n{r['emb']}")


=== C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
--- LLM only:
Which student funding scheme or country do you mean? “Income band 2” is used by several programs, and the amount and payment schedule vary. Please share the scheme name, institution, or a link.
--- BM25 RAG ['GO-04', 'GO-02', 'GO-01']:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]
--- Embedding RAG ['GO-04', 'GO-03', 'GO-15']:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**—one in **October** and one in **February**. [1]



=== C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.
--- LLM only:
The appeal deadline and form depend on the **grant scheme, funding body, and country**. They should be stated in your refusal/decision letter—often the deadline runs from the date you were notified, not the date you read it.

Please tell me:

- the grant or funding body/scheme;
- your country or jurisdiction; and
- the date and wording of the refusal (you can redact personal details).

I can then identify the exact appeal deadline and form. Don’t rely on a general deadline or miss the date while checking.
--- BM25 RAG ['GO-05', 'GO-08', 'GO-12']:
You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**. [2][3]
--- Embedding RAG ['GO-08', 'GO-12', 'GO-11']:
You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305


=== C-03: What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
--- LLM only:
Which need-based grant or school are you referring to? GPA requirements vary by program; some need-based grants have no minimum GPA initially but require satisfactory academic progress, often around a 2.0–2.5 GPA.
--- BM25 RAG ['GO-16', 'GO-02', 'GO-06']:
The minimum GPA is **2.67 on a 4.0 scale**. [2]
--- Embedding RAG ['GO-02', 'GO-20', 'GO-05']:
The minimum GPA is **2.67 on a 4.0 scale**. [1]



=== C-04: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
REFERENCE: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.
--- LLM only:
Kazakhstan formally decided to move from Cyrillic to a Latin-based Kazakh alphabet on **26 October 2017**, when President Nursultan Nazarbayev issued the relevant decree. (The intention had been announced earlier, in **2012**.)

The transition was initially supposed to be completed by **2025**, but the timetable was subsequently extended. It is now scheduled to be completed by **2031**.
--- BM25 RAG ['KZ-43', 'KZ-14', 'KZ-06']:
Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition, initially planned for 2025, is now due to finish in **2031**. [1][2]
--- Embedding RAG ['KZ-14', 'KZ-43', 'KZ-05']:
Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition is 


=== C-05: When was Narxoz University founded, and what was it called at the start?
REFERENCE: In 1963, as the Almaty Institute of National Economy; it took its current name in 2016.
--- LLM only:
Narxoz University was founded in **1963** as the **Alma-Ata Institute of National Economy**.
--- BM25 RAG ['KZ-44', 'KZ-11', 'KZ-07']:
Narxoz University was founded in 1963 as the **Almaty Institute of National Economy**. [1]
--- Embedding RAG ['KZ-44', 'KZ-05', 'KZ-14']:
Narxoz University was founded in 1963, originally called the Almaty Institute of National Economy. [1]



=== C-06: I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
--- LLM only:
The amount depends mainly on your **country, school, program, income/assets, enrollment status, and whether you’re considered independent**.

If you mean the **United States**:

- **Federal Pell Grant:** Generally **not available for students in a master’s degree program**. It is primarily for undergraduates, with limited exceptions for certain post-baccalaureate teacher-certification programs.
- **Graduate need-based grants:** There is no standard federal grant for most master’s students. You may qualify for:
  - University or department grants
  - State grants
  - Fellowships or assistantships
  - Federal Work-Study
  - TEACH Grant, if you’re preparing to teach in an eligible high-need field/school
- **Federal loans:** FAFSA may still make you eligible for Direct Unsubsidized

Marks from reading the printed answers (✓ agrees with `reference`, ✗ does not):

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asks which scheme; no number | ✓ 150,000 tenge, two instalments Oct/Feb [1] | ✓ 150,000, 2 × 75,000, Oct/Feb [1] |
| C-02 | ✗ asks for scheme/country; no 14 days, no form | ✓ 14 calendar days, GR-305 [2][3] | ✓ 14 calendar days, GR-305 [1][2] |
| C-03 | ✗ "often around 2.0–2.5" and asks for the program | ✓ 2.67 on 4.0 [2] | ✓ 2.67 on 4.0 [1] |
| C-04 | ✓ October 2017 (26 Oct), 2025 → 2031 | ✓ October 2017, 2025 → 2031 [1][2] | ✓ October 2017, due 2031 [1][2] |
| C-05 | ✓ 1963, "Alma-Ata Institute of National Economy" | ✓ 1963, Almaty Institute of National Economy [1] | ✓ 1963, Almaty Institute of National Economy [1] |
| C-06 | ✗ answers about US Pell/FAFSA, invented figures ($20,500 loans) | ✓ "I do not know" [3] | ✓ "I don't know the amount" [1] |
| **agrees with `reference`** | **2/6** | **6/6** | **6/6** |

Notes: the LLM-only system fails exactly on the fictional handbook (C-01, C-02, C-03, C-06) and is fine on the Wikipedia questions (C-04, C-05): there retrieval added nothing the model did not already know. Both RAG systems said "I do not know" for C-06, the only correct response.

## C3 — question 1 of 3: C-03

Why chosen: the archived GO-05 states `minimum GPA ... 3.0` in the same words as the question; the question tests whether the wrong rule gets in.

Sections 1–3 are printed by the next cell. Sections 4–5:

**4. How retrieval changed the answer:** **more accurate and grounded.** LLM only had no way to know the handbook and guessed "often around 2.0–2.5". Both RAG systems answered 2.67 on the 4.0 scale and cited the right block. The two retrievers got there differently: BM25 top 3 was GO-16 (15.80), GO-02 (14.06), GO-06 (13.85), so the right chunk was only 2nd, and the archived GO-05 did **not** make the top 3. EmbeddingGemma's top 3 was GO-02 (0.6286), GO-20 (0.5691), GO-05 (0.5451): the archived chunk was in the context sent to the model as block [3].

**5. What was missed, and was it a retrieval problem or a generation problem?** Nothing was missed in the final answer. The risk was there, though: in the embedding context both GO-02 (2.67, block [1]) and GO-05 (ARCHIVED, 3.0, block [3]) were present, and the model chose block [1] and answered 2.67. So retrieval brought a distractor and **generation handled it correctly**, because GO-05 literally says "no longer in force". With BM25 the distractor did not enter, but irrelevant chunks did (GO-16 ranked first, GO-06 third).

In [11]:
def show(qid):
    q = qa_by[qid]; r = c2[qid]
    print(q["question"], "\nREFERENCE:", q["reference"], "\nrelevant:", q["relevant"])
    print("\n1. BM25 top 3:", [(i, round(s, 4)) for i, s in bm25_top(q["question"])])
    print("2. EmbeddingGemma top 3:", [(i, round(s, 4)) for i, s in emb_top(q["question"])])
    print("\n3. Context sent to the model (BM25 RAG):\n" + context_block(r["bm25_ids"]))
    print("\n   Context sent to the model (Embedding RAG):\n" + context_block(r["emb_ids"]))
    print("\nLLM only:", r["llm"], "\n\nBM25 RAG:", r["bm25"], "\n\nEmbedding RAG:", r["emb"])
    for name in ("bm25_ids", "emb_ids"):
        print(f"{name}: relevant chunks in context ->", [i for i in q["relevant"] if i in r[name]],
              "| missing ->", [i for i in q["relevant"] if i not in r[name]])
show("C-03")

What is the minimum GPA needed to qualify for the need-based grant? 
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.) 
relevant: ['GO-02']

1. BM25 top 3: [('GO-16', 15.7985), ('GO-02', 14.0587), ('GO-06', 13.8516)]
2. EmbeddingGemma top 3: [('GO-02', 0.6286), ('GO-20', 0.5691), ('GO-05', 0.5451)]

3. Context sent to the model (BM25 RAG):
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or

## C3 — question 2 of 3: C-06

Why chosen: no answer in the corpus (GO-20 only says master's students have a separate scheme). The only correct answer is that the system does not know.

Sections 1–3 are printed by the next cell. Sections 4–5:

**4. How retrieval changed the answer:** **more grounded, and the only way to be right.** LLM only gave a long answer about the US Pell Grant, FAFSA and "$20,500 per academic year" of loans: confident, plausible and unrelated to this grant office. Both RAG systems said they do not know. BM25 top 3: GO-16 (16.9958), GO-01 (14.1725), GO-20 (13.7202). EmbeddingGemma top 3: GO-20 (0.6664), GO-16 (0.6161), GO-01 (0.5539). In both runs GO-20 was in the context.

**5. What was missed, and was it a retrieval problem or a generation problem?** The amount was not missed by either system, because it is not in the corpus. This was neither failure: retrieval returned the one chunk that explains why there is no answer, and generation followed the system message ("If the answer cannot be found in the context, say that you do not know"). BM25 said: "I do not know. The context says that master's students are supported by a separate scheme, but it does not state its grant amount. [3]"; embeddings: "I don't know the amount. Master's students are not covered by this scheme and should ask their department about its separate rules. [1]". The embedding answer is slightly better grounded because GO-20 was ranked first.

In [12]:
show("C-06")

I am a master's student. How much need-based grant can I get? 
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe. 
relevant: ['GO-20']

1. BM25 top 3: [('GO-16', 16.9958), ('GO-01', 14.1725), ('GO-20', 13.7202)]
2. EmbeddingGemma top 3: [('GO-20', 0.6664), ('GO-16', 0.6161), ('GO-01', 0.5539)]

3. Context sent to the model (BM25 RAG):
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] About the need-based study grant 2026
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay f

## C3 — question 3 of 3: C-02

Why chosen: the answer needs two facts (14 calendar days and form GR-305) spread over GO-08, GO-12 and GO-23 (the last in Russian), and the corpus has three similar form names. This is also where BM25 retrieval was worse than expected.

Sections 1–3 are printed by the next cell. Sections 4–5:

**4. How retrieval changed the answer:** **more accurate and specific.** LLM only could not answer ("depends on the scheme, country ... tell me") and gave no number and no form. Both RAG systems answered "14 calendar days ... Form GR-305". But BM25's context was worse: top 3 was **GO-05 (17.2414)**, GO-08 (14.5287), GO-12 (13.5735); the archived 2025 scheme came **first** and took one of three slots, although it is irrelevant to appeals. EmbeddingGemma: GO-08 (0.6029), GO-12 (0.5591), GO-11 (0.5343), a "missing document" chunk, also not needed, so both contexts held one useless chunk.

**5. What was missed, and was it a retrieval problem or a generation problem?** GO-23 (the Russian summary of the appeal rule) was in neither context (`missing -> ['GO-23']` for both). That is a **retrieval** miss, but harmless here because GO-08 and GO-12 hold both needed facts, and the model produced the correct answer from them. The K = 1/3/8 experiment in the final questions shows what happens when the fact is not in the context.

In [13]:
show("C-02")

My grant application was refused. How long do I have to appeal, and which form do I use? 
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305. 
relevant: ['GO-08', 'GO-12', 'GO-23']

1. BM25 top 3: [('GO-05', 17.2414), ('GO-08', 14.5287), ('GO-12', 13.5735)]


2. EmbeddingGemma top 3: [('GO-08', 0.6029), ('GO-12', 0.5591), ('GO-11', 0.5343)]

3. Context sent to the model (BM25 RAG):
[1] Archived: the 2025 scheme (no longer in force)
ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

[2] Form GR-305: appeals
Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

[3] How to appeal a decision
A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, usi

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 scores by overlap of exact tokens (term frequency, rarity via idf, length normalisation); embeddings map text to dense vectors and score by closeness in meaning, so they work without shared words. BM25 is lexical, embeddings are semantic.

**2. When can BM25 perform better than embeddings?**

> When the query hinges on an exact, rare string: a form code like `GR-305` or `GR-210`, an id, a name, a number. BM25 gives that token a high idf, while the embedding model tends to blur near-identical codes. It also needs no model at all.

**3. When can embeddings perform better than BM25?**

> When query and answer use different words for the same thing: synonyms ("poor" vs "low-income", "year off" vs "academic leave"), paraphrases, or another language (the Kazakh query Q-03 over mostly English chunks). BM25 matches nothing in those cases.

**4. Why is cosine similarity used with embeddings?**

> It compares the direction of two vectors and ignores their length, so a long and a short text about the same thing still score as similar. With normalised vectors it is just a dot product, which is cheap to compute against every chunk.

**5. What information is actually sent to an LLM in a RAG system?**

> A prompt made of text: the instructions, the retrieved chunks (as numbered blocks) and the question (see the printed C1 prompt). Nothing else.

**6. Does the LLM search the original documents itself?**

> No. The code searches (BM25 or embeddings) and pastes the winners into the prompt; the model never opens a document and can only use what was pasted, plus what it learned in training.

**7. What happens if retrieval returns an irrelevant chunk?**

> The model cannot know it is irrelevant except by reading it. Best case it ignores it or says "I do not know"; worst case it builds a confident, cited answer from the wrong text (for example the archived 3.0 GPA in GO-05 for C-03). See the C3 analysis for what happened in my runs.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> A bigger K raises the chance that the needed chunk is included (more recall), which helps when the answer is spread over several chunks (C-02 needs GO-08, GO-12 and GO-23). But every extra chunk adds tokens and noise; distractors (the archived rule, similar form names) can mislead the model or bury the right fact. The next cell runs C-02 with K = 1, 3 and 8, and the results were:
> - **K = 1** (GO-08 only, 157 prompt tokens): "You use Form GR-305 to appeal the refusal. The context does not specify how long you have to appeal." The fact "14 calendar days" is in GO-12, which was not retrieved: a **retrieval** failure caused by K being too small, and the model correctly said what it did not know instead of inventing it.
> - **K = 3** (GO-08, GO-12, GO-11; 341 tokens): correct, "14 calendar days ... Form GR-305". GO-11 is an unrelated distractor.
> - **K = 8** (all three relevant chunks plus GO-06, GO-09, GO-17, GO-21; 802 tokens): still correct, but about 2.4x the tokens of K = 3 and five irrelevant chunks. Here more context did not hurt, it only cost more, and each irrelevant chunk is a chance to mislead the model. So K = 1 → 3 improved the answer (recall), 3 → 8 added only cost for this question.

In [14]:
q = qa_by["C-02"]
for k in (1, 3, 8):
    top = [i for i, _ in emb_top(q["question"], k)]
    a, used = ask(build_messages(q["question"], top))
    print(f"--- K={k}  chunks={top}  relevant in context={[i for i in q['relevant'] if i in top]}  prompt_tokens={used}\n{a}\n")

--- K=1  chunks=['GO-08']  relevant in context=['GO-08']  prompt_tokens=157
You use **Form GR-305** to appeal the refusal. The context does not specify how long you have to appeal. [1]



--- K=3  chunks=['GO-08', 'GO-12', 'GO-11']  relevant in context=['GO-08', 'GO-12']  prompt_tokens=341
You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**. [2]



--- K=8  chunks=['GO-08', 'GO-12', 'GO-11', 'GO-23', 'GO-06', 'GO-09', 'GO-17', 'GO-21']  relevant in context=['GO-08', 'GO-12', 'GO-23']  prompt_tokens=802
You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**. [1][2]



**9. Is a vector database required to build a RAG system?**

> No. My notebook kept the vectors in a plain numpy array in memory (`doc_emb`, 74 × 768) and searched it by computing `model.similarity` against every row. A vector database only matters at scale (millions of chunks, persistence, approximate nearest-neighbour search).

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> For this corpus **EmbeddingGemma worked better**. hit@3 over my ten queries: **BM25 6/10, EmbeddingGemma 10/10**. BM25 failed on Q-02, Q-04, Q-07 and Q-10: a synonym ("poor" vs "low-income"), a wrong, archived chunk coming first (GO-05 for the GPA query), a Russian chunk (GO-15) and a paraphrase ("year off" vs "academic leave"). In C2 the two RAG systems **both agree with `reference` on 6/6** (LLM only 2/6), so with top 3 the end answers did not differ, even where BM25 retrieved worse chunks (C-02, C-03: GO-05, GO-16 in the context). The evidence for embeddings is therefore the retrieval quality (10/10 vs 6/10), not the final answers on these six questions, which happen to contain strong lexical anchors (`GR-305`, `2.67`, `Narxoz`).

---
# Conclusion

> Next I would build hybrid retrieval (BM25 for exact codes, embeddings for paraphrases and other languages, merged with reciprocal rank fusion), because the two fail in opposite places in this corpus. I would also mark archived chunks in metadata and filter them out, and keep the prompt rule that the model must say "I do not know" when the context lacks the answer, because those are the failures that retrieval alone does not fix. Final figures should be quoted from my own run outputs.